In [0]:
%run ./00_Config_Setup

In [0]:
print("Processando Silver Yellow Taxi...")

df_yellow = spark.table(f"{catalog}.{schema}.bronze_yellow")

# Normalizar nomes de colunas para minúsculo
df_yellow = df_yellow.toDF(*[c.lower() for c in df_yellow.columns])

# Seleção Dinâmica de data para Yellow (comumente tpep_pickup_datetime)
cols_yellow = df_yellow.columns
yellow_pickup_expr = F.expr("try_to_timestamp(tpep_pickup_datetime)") if "tpep_pickup_datetime" in cols_yellow else F.expr("try_to_timestamp(pickup_datetime)")

# Lógica para Dropoff
if "dropoff_datetime" in cols_yellow:
    dropoff_col = F.expr("try_to_timestamp(dropoff_datetime)")
elif "dropoff_date" in cols_yellow:
    dropoff_col = F.expr("try_to_timestamp(dropoff_date)")
else:
    # se não existir nenhuma, cria uma coluna vazia
    dropoff_col = F.lit(None).cast("timestamp")


df_silver_yellow = df_yellow.select(
    yellow_pickup_expr.alias("pickup_datetime"),
    dropoff_col.alias("dropoff_datetime"),
    F.lit("Yellow Taxi").alias("taxi_type")
).filter(F.col("pickup_datetime").isNotNull())

# Salva a tabela Silver Yellow
df_silver_yellow.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.{schema}.silver_yellow")

print("Tabela silver_yellow criada com sucesso.")

In [0]:
print("Processando Silver FHV/Uber/Lyft...")

# Seta o dataframe
df_fhv = spark.table(f"{catalog}.{schema}.bronze_fhv")

# normaliza nomes de colunas
df_fhv = df_fhv.toDF(*[c.lower() for c in df_fhv.columns])

# Faz o tratamento de dados
cols = df_fhv.columns
pickup_expr = F.expr("try_to_timestamp(pickup_datetime)") if "pickup_datetime" in cols else F.expr("try_to_timestamp(pickup_date)")

df_silver_fhv = df_fhv.select(
    pickup_expr.alias("pickup_datetime"),
#    F.col("pulocationid").cast("integer"), # Algumas tabelas FHV possuem (virará null se não existir)
    F.lit("App (Uber/Lyft)").alias("taxi_type")
).filter(F.col("pickup_datetime").isNotNull())

# salva a tabela silver FHV
df_silver_fhv.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.silver_fhv")

print("Tabela silver_fhv criada com sucesso.")